# AURA — Entity model (date, time, service, name)

Fine-tunes **XLM-RoBERTa-base** to tag the details in a customer message (BIO tags):

`kal 3 baje haircut book kar do, mera naam Ali hai` → **kal** = DATE · **3 baje** = TIME · **haircut** = SERVICE · **Ali** = NAME

This completes the proposal's NLU layer: the intent model says *what* the customer wants, this model finds *the details*.
AURA then turns the DATE/TIME text into real dates with its rule-based parser.

**Test sets** (never used for training): `test` (the original 136 messages) and `test_noisy` (the same messages re-spelled).
Scores are **entity-level** (seqeval): an entity counts only if its type *and* exact words are right.

### How to run
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**
3. When asked, upload the **5 files** from `AURA/training/data_ner/`: `train.json`, `val.json`, `test.json`, `test_noisy.json`, `labels.json`
4. Click **Allow** for Google Drive. Keep this tab open until the last cell prints **DONE** (about 10–15 minutes).

## 1. Setup

In [ ]:
!pip install -q -U transformers accelerate seqeval

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

## 2. Upload the dataset

In [ ]:
import json, os
from google.colab import files

FILES = ["train.json", "val.json", "test.json", "test_noisy.json", "labels.json"]
missing = [f for f in FILES if not os.path.exists(f)]
if missing:
    print("Please upload (you can select all 5 at once):", missing)
    files.upload()

data = {}
for f in FILES[:4]:
    with open(f, encoding="utf-8") as fh:
        data[f[:-5]] = json.load(fh)
    print(f"{f:16} {len(data[f[:-5]])} messages")
LABELS = json.load(open("labels.json"))
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
print("Labels:", LABELS)

## 3. Tokenize (each word's tag goes on its first sub-word)

In [ ]:
from transformers import AutoTokenizer, set_seed

SEED, MODEL_NAME, MAX_LEN = 42, "xlm-roberta-base", 96
set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class NerDataset(torch.utils.data.Dataset):
    def __init__(self, rows):
        self.enc = tokenizer([r["tokens"] for r in rows], is_split_into_words=True,
                             truncation=True, max_length=MAX_LEN)
        self.labels = []
        for i, r in enumerate(rows):
            word_ids, previous, labels = self.enc.word_ids(i), None, []
            for w in word_ids:
                if w is None or w == previous:
                    labels.append(-100)              # special tokens and later sub-words are ignored
                else:
                    labels.append(label2id[r["ner_tags"][w]])
                previous = w
            self.labels.append(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

ds = {name: NerDataset(rows) for name, rows in data.items()}
print({k: len(v) for k, v in ds.items()})

## 4. Train

In [ ]:
import math
import numpy as np
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report
from transformers import (AutoModelForTokenClassification, TrainingArguments, Trainer,
                          DataCollatorForTokenClassification, EarlyStoppingCallback)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id)

def to_tags(predictions, label_ids):
    true, pred = [], []
    for p_row, l_row in zip(predictions, label_ids):
        t, p = [], []
        for p_id, l_id in zip(p_row, l_row):
            if l_id != -100:
                t.append(id2label[int(l_id)]); p.append(id2label[int(p_id)])
        true.append(t); pred.append(p)
    return true, pred

def compute_metrics(out):
    true, pred = to_tags(np.argmax(out.predictions, -1), out.label_ids)
    return {"precision": precision_score(true, pred), "recall": recall_score(true, pred), "f1": f1_score(true, pred)}

EPOCHS, BATCH = 10, 16
warmup_steps = int(0.1 * math.ceil(len(ds["train"]) / BATCH) * EPOCHS)

args = TrainingArguments(
    output_dir="checkpoints_ner", num_train_epochs=EPOCHS, learning_rate=3e-5,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=32, warmup_steps=warmup_steps,
    weight_decay=0.01, eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="f1", greater_is_better=True,
    fp16=torch.cuda.is_available(), logging_strategy="epoch", report_to="none", seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["val"],
                  data_collator=DataCollatorForTokenClassification(tokenizer), compute_metrics=compute_metrics,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
trainer.train()
print("Best validation:", trainer.evaluate())

## 5. Evaluate on the test sets

In [ ]:
results, reports, errors = {}, {}, []
for name in ["test", "test_noisy"]:
    out = trainer.predict(ds[name])
    true, pred = to_tags(np.argmax(out.predictions, -1), out.label_ids)
    results[name] = {"messages": len(true), "precision": precision_score(true, pred),
                     "recall": recall_score(true, pred), "f1": f1_score(true, pred)}
    reports[name] = classification_report(true, pred, digits=3)
    print(f"=== {name}: F1 {results[name]['f1']:.3f}  (precision {results[name]['precision']:.3f}, "
          f"recall {results[name]['recall']:.3f})\n{reports[name]}")
    for row, t, p in zip(data[name], true, pred):
        if t != p:
            errors.append({"set": name, "tokens": row["tokens"], "true": t, "predicted": p})
print(f"{len(errors)} messages with at least one tagging difference. First 10:")
for e in errors[:10]:
    diff = [f"{w}: {a}->{b}" for w, a, b in zip(e["tokens"], e["true"], e["predicted"]) if a != b]
    print(f"  [{e['set']}] {' '.join(e['tokens'])}\n      {diff}")

## 6. Save to Google Drive (checked: Drive connected, weights saved, saved copy really trained)

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
assert os.path.ismount("/content/drive"), "Google Drive is NOT connected - run this cell again and click Allow"

SAVE_DIR = "/content/drive/MyDrive/AURA/entity_model"
os.makedirs(SAVE_DIR, exist_ok=True)
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
with open(f"{SAVE_DIR}/results_ner.json", "w") as f:
    json.dump(results, f, indent=2)
with open(f"{SAVE_DIR}/classification_report_ner.txt", "w") as f:
    f.write("\n\n".join(f"{k}\n{v}" for k, v in reports.items()))
with open(f"{SAVE_DIR}/errors_ner.json", "w", encoding="utf-8") as f:
    json.dump(errors, f, ensure_ascii=False, indent=1)

weights = os.path.join(SAVE_DIR, "model.safetensors")
size_gb = os.path.getsize(weights) / 1e9 if os.path.exists(weights) else 0
assert size_gb > 0.5, f"model.safetensors missing or too small ({size_gb:.2f} GB) - re-run this cell"

# Reload the SAVED copy and score it: catches saving an untrained model (e.g. after a Colab reset)
_m = AutoModelForTokenClassification.from_pretrained(SAVE_DIR).to(model.device).eval()
_t = Trainer(model=_m, args=args, eval_dataset=ds["val"],   # args evaluate every epoch, so a dataset is required
             data_collator=DataCollatorForTokenClassification(tokenizer))
_out = _t.predict(ds["test"])
saved_f1 = f1_score(*to_tags(np.argmax(_out.predictions, -1), _out.label_ids))
print(f"Saved to {SAVE_DIR} (model.safetensors = {size_gb:.2f} GB). Saved model F1 on test: {saved_f1:.3f}")
assert saved_f1 > 0.5, "The SAVED model is not the trained one - re-run Runtime > Run all"
del _m, _t
!ls -lh "$SAVE_DIR"

## 7. Try your own messages

In [ ]:
def tag(text):
    words = text.split()
    enc = tokenizer(words, is_split_into_words=True, return_tensors="pt", truncation=True,
                    max_length=MAX_LEN).to(model.device)
    with torch.no_grad():
        ids = model(**enc).logits.argmax(-1)[0].tolist()
    seen, out = set(), []
    for pos, w in enumerate(enc.word_ids()):
        if w is not None and w not in seen:
            seen.add(w); out.append((words[w], id2label[ids[pos]]))
    return [(w, t) for w, t in out if t != "O"]

for msg in ["kal 3 baje haircut book kar do", "mera naam Ali Khan hai", "parso sham 6 bje facial krwana he",
            "Friday ko dental checkup ki appointment chahiye 11 am", "Sara ke liye beard trim 15th November",
            "booking cancle krni he kl wali"]:
    print(tag(msg), "<-", msg)

## 8. Finish the upload to Drive

In [ ]:
from google.colab import drive
assert os.path.ismount("/content/drive"), "Google Drive is not connected - re-run section 6 first"
print("Uploading to Google Drive, please wait (can take 5-10 minutes)...")
drive.flush_and_unmount()
print("DONE - download the folder My Drive/AURA/entity_model")